# Validate soundscape macro-AUC vs leaderboard

## Hypothesis

Training on focal **`train_audio/`** clips (single-species recordings) while validating on held-out focal clips measures performance on a distribution that differs sharply from the competition objective. The hidden **`test_soundscapes/`** data are multi-label, multi-species field recordings with noise and overlapping birds. Local ROC-AUC computed only on focal validation can therefore be **optimistically biased** and **poorly correlated** with public leaderboard (LB) macro-AUC.

## What we test here

If we instead score archived checkpoints on **`train_soundscapes/`** with labels from **`train_soundscapes_labels.csv`** (same windowing and row-id convention as a Kaggle submission), the resulting **soundscape macro-AUC** should **correlate** with your known LB scores across submissions **when preprocessing matches inference**. High rank correlation together with similar absolute values would support refactoring the agent to use soundscape-based validation during training.

This notebook loads historical **`model.keras`** files under **`submission_archive/`**, runs soundscape inference with fixed mel settings aligned to training, compares predictions to the merged multi-label targets, and builds a results table with a manual **`actual_lb_score`** column for correlation analysis.

## Setup — imports, paths, hyperparameters

Match training / Kaggle inference defaults: **32 kHz** mono mel spectrograms resized to **`(N_MELS, N_FRAMES)`**, five-second segments.

In [1]:
from __future__ import annotations

import warnings
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import average_precision_score, roc_auc_score
from scipy.stats import spearmanr

warnings.filterwarnings("ignore", category=UserWarning)

# Resolve project root (notebook lives under notebooks/)
_HERE = Path.cwd().resolve()
PROJECT_ROOT = _HERE.parent if _HERE.name == "notebooks" else _HERE

DATA_DIR = PROJECT_ROOT / "data"
ARCHIVE_ROOT = PROJECT_ROOT / "submission_archive"
TRAIN_SOUNDSCAPES_DIR = DATA_DIR / "train_soundscapes"
LABELS_CSV = DATA_DIR / "train_soundscapes_labels.csv"
SAMPLE_SUB_CSV = DATA_DIR / "sample_submission.csv"

SR = 32000
CLIP_SECONDS = 5
N_MELS = 64
N_FRAMES = 128
N_FFT = 1024
HOP_LENGTH = 512

N_WINDOWS = 12
TARGET_SECONDS = N_WINDOWS * CLIP_SECONDS
TARGET_SAMPLES = int(SR * TARGET_SECONDS)
SEG_SAMPLES = int(SR * CLIP_SECONDS)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TensorFlow:", tf.__version__)
print("train_soundscapes dir:", TRAIN_SOUNDSCAPES_DIR)
print("train_soundscapes exists:", TRAIN_SOUNDSCAPES_DIR.exists())

/Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


PROJECT_ROOT: /Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project
TensorFlow: 2.20.0
train_soundscapes dir: /Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/data/train_soundscapes
train_soundscapes exists: True


## Discover archived models

Walk **`submission_archive/`** for every **`model.keras`**. The parent folder name is treated as the submission label (e.g. `sub_7_30.04_v1.6_sec_label_1_0`).

In [2]:
if not ARCHIVE_ROOT.is_dir():
    raise FileNotFoundError(f"Missing archive root: {ARCHIVE_ROOT}")

found_paths = sorted(ARCHIVE_ROOT.rglob("model.keras"))
discovered_df = pd.DataFrame(
    [{"submission_label": p.parent.name, "model_path": p} for p in found_paths]
)

try:
    from IPython.display import display
except ImportError:
    display = lambda df: print(df.to_string())  # noqa: E731

print(f"Found {len(discovered_df)} model.keras file(s).")
display(discovered_df)

Found 5 model.keras file(s).


,submission_label,model_path
0,sub_5_29.04_v1.4,/Users/enricozafiris/Desktop/Catolica/T4 Advan...
1,sub_6_29.04_v1.5,/Users/enricozafiris/Desktop/Catolica/T4 Advan...
2,sub_7_30.04_v1.6_sec_label_1_0,/Users/enricozafiris/Desktop/Catolica/T4 Advan...
3,sub_8_30.04_v1.7_sec_label_weighted_0_5,/Users/enricozafiris/Desktop/Catolica/T4 Advan...
4,sub_9_30.04_v2.0_pretrained,/Users/enricozafiris/Desktop/Catolica/T4 Advan...


## Configure which models to evaluate

**Default:** evaluate every discovered checkpoint.

To restrict evaluation, set **`MODEL_PATHS_TO_EVAL`** to an explicit list of `Path` objects and comment out entries you do not want. Leaving **`MODEL_PATHS_TO_EVAL = None`** runs all rows from **`discovered_df`**.

In [3]:
# Default: evaluate all discovered archived model.keras files
MODEL_PATHS_TO_EVAL = None

# Optional: restrict to a subset by listing explicit paths
# MODEL_PATHS_TO_EVAL = [
#     PROJECT_ROOT / "submission_archive" / "sub_5_29.04_v1.4" / "model.keras",
# ]

if MODEL_PATHS_TO_EVAL is None:
    eval_paths = list(discovered_df["model_path"])
else:
    eval_paths = [Path(p).resolve() for p in MODEL_PATHS_TO_EVAL]

missing = [str(p) for p in eval_paths if not p.is_file()]
if missing:
    print("WARNING: missing files:", missing)
eval_paths = [p for p in eval_paths if p.is_file()]

print(f"Models queued for evaluation: {len(eval_paths)}")
for p in eval_paths:
    print(" -", p)

Models queued for evaluation: 1
 - /Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/submission_archive/sub_5_29.04_v1.4/model.keras


## Species columns (234-class order)

Use **`sample_submission.csv`** column order (excluding **`row_id`**) so predictions align with competition labeling.

In [4]:
sample_sub = pd.read_csv(SAMPLE_SUB_CSV)
species_cols = [c for c in sample_sub.columns if c != "row_id"]
species_to_idx = {s: i for i, s in enumerate(species_cols)}
n_species = len(species_cols)
print("Species columns:", n_species)

Species columns: 234


## Build soundscape ground truth

For each unique **`(filename, start, end)`** triple, union every semicolon-separated taxon code from **`primary_label`**. **`row_id`** matches submission format: **`<stem>_<end_sec>`** with **`end_sec`** from the **`end`** timestamp.

In [6]:
raw = pd.read_csv(LABELS_CSV)


def _tokens_primary(val: object) -> set[str]:
    if pd.isna(val) or val == "":
        return set()
    return {t.strip() for t in str(val).split(";") if t.strip()}


def _merge_labels(series: pd.Series) -> set[str]:
    out: set[str] = set()
    for v in series:
        out |= _tokens_primary(v)
    return out


grp = (
    raw.groupby(["filename", "start", "end"], sort=False)["primary_label"]
    .agg(_merge_labels)
    .reset_index()
)

grp["end_sec"] = pd.to_timedelta(grp["end"]).dt.total_seconds().astype(int)
grp["row_id"] = grp["filename"].str.replace(".ogg", "", regex=False) + "_" + grp["end_sec"].astype(
    str
)

unknown_codes = set()
rows = []
for _, r in grp.iterrows():
    vec = np.zeros(n_species, dtype=np.float32)
    for code in r["primary_label"]:
        j = species_to_idx.get(code)
        if j is None:
            unknown_codes.add(code)
        else:
            vec[j] = 1.0
    rows.append((r["row_id"], vec))

if unknown_codes:
    print("WARNING: label codes not in sample_submission (first 20):", sorted(unknown_codes)[:20])

y_true_df = pd.DataFrame(
    [v for _, v in rows],
    index=[rid for rid, _ in rows],
    columns=species_cols,
).sort_index()
if not y_true_df.index.is_unique:
    y_true_df = y_true_df.groupby(level=0).max()
y_true_df.index.name = "row_id"

print("Ground-truth rows:", len(y_true_df))
print("Species with ≥1 positive:", int((y_true_df.sum(axis=0) > 0).sum()))

Ground-truth rows: 739
Species with ≥1 positive: 75


## Inference — mirror Kaggle soundscape submission

Load mono audio at **`SR`**, **pad/truncate to 60 s**, slice **twelve** **`CLIP_SECONDS`** windows, compute mel → dB → **`tf.image.resize`** to **`(N_MELS, N_FRAMES)`**, **`model.predict`**. **`row_id`** uses **`end_sec = (segment_index + 1) * CLIP_SECONDS`** (5, 10, …, 60).

Progress prints every **10** completed soundscape files.

In [7]:
def soundscapes_inference_df(
    model: tf.keras.Model,
    ogg_paths: list[Path],
    species_columns: list[str],
    *,
    progress_every: int = 10,
) -> pd.DataFrame:
    all_rows: list[dict] = []
    for fi, fpath in enumerate(ogg_paths):
        if fi > 0 and fi % progress_every == 0:
            print(f"  … {fi}/{len(ogg_paths)} soundscapes")
        name = fpath.stem
        y_full, _ = librosa.load(str(fpath), sr=SR, mono=True)
        if len(y_full) > TARGET_SAMPLES:
            y_full = y_full[:TARGET_SAMPLES]
        elif len(y_full) < TARGET_SAMPLES:
            y_full = np.pad(y_full, (0, TARGET_SAMPLES - len(y_full)))

        batch = np.zeros((N_WINDOWS, N_MELS, N_FRAMES, 1), dtype=np.float32)
        for wi in range(N_WINDOWS):
            start = wi * SEG_SAMPLES
            seg = y_full[start : start + SEG_SAMPLES]
            mel = librosa.feature.melspectrogram(
                y=seg,
                sr=SR,
                n_mels=N_MELS,
                n_fft=N_FFT,
                hop_length=HOP_LENGTH,
                power=2.0,
            )
            mel_db = librosa.power_to_db(mel, ref=np.max)
            mel_r = (
                tf.image.resize(mel_db[..., np.newaxis], (N_MELS, N_FRAMES))
                .numpy()
                .astype(np.float32)
            )
            batch[wi] = mel_r

        preds = model.predict(batch, verbose=0).astype(np.float32)
        for wi in range(N_WINDOWS):
            end_sec = (wi + 1) * CLIP_SECONDS
            row = {"row_id": f"{name}_{end_sec}"}
            for col, p in zip(species_columns, preds[wi]):
                row[col] = float(p)
            all_rows.append(row)

    return pd.DataFrame(all_rows)

## Metric helpers

**Macro-AUC:** Among species with at least **`min_pos`** positives in aligned **`y_true`**, compute **`roc_auc_score(..., average='macro')`**. Columns without both negatives and positives are skipped (undefined per-class AUC).

**Other summaries:** **top-5 hit rate** on rows with ≥1 true label (any true species in top-5 predictions), and **macro mAP** via **`average_precision_score`**.

In [8]:
def macro_auc_filtered(y_true: np.ndarray, y_pred: np.ndarray, *, min_pos: int) -> float:
    keep = y_true.sum(axis=0) >= min_pos
    if not np.any(keep):
        return float("nan")
    yt = y_true[:, keep]
    yp = y_pred[:, keep]
    usable = []
    for j in range(yt.shape[1]):
        col = yt[:, j]
        if col.min() == 0 and col.max() == 1:
            usable.append(j)
    if not usable:
        return float("nan")
    usable = np.array(usable, dtype=int)
    return float(
        roc_auc_score(yt[:, usable], yp[:, usable], average="macro")
    )


def top5_hit_rate(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    hits = 0
    eligible = 0
    for i in range(len(y_true)):
        true_idx = np.flatnonzero(y_true[i] > 0.5)
        if true_idx.size == 0:
            continue
        eligible += 1
        top5 = np.argsort(-y_pred[i])[:5]
        if np.any(np.isin(top5, true_idx)):
            hits += 1
    return hits / eligible if eligible else float("nan")


def macro_map(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    try:
        return float(
            average_precision_score(y_true, y_pred, average="macro")
        )
    except ValueError:
        return float("nan")


# Use only soundscape files that are needed by y_true_df row_ids (faster than scanning all .ogg)
required_stems = {rid.rsplit("_", 1)[0] for rid in y_true_df.index}
ogg_files = sorted(
    [
        TRAIN_SOUNDSCAPES_DIR / f"{stem}.ogg"
        for stem in required_stems
        if (TRAIN_SOUNDSCAPES_DIR / f"{stem}.ogg").is_file()
    ]
)

missing_soundscapes = sorted(
    [
        stem
        for stem in required_stems
        if not (TRAIN_SOUNDSCAPES_DIR / f"{stem}.ogg").is_file()
    ]
)
if missing_soundscapes:
    print(
        f"WARNING: missing {len(missing_soundscapes)} labeled soundscape file(s); "
        "metrics will be computed on available overlap only."
    )

print(
    f"Labeled windows in y_true_df: {len(y_true_df)} | "
    f"required soundscape files: {len(required_stems)} | "
    f"found for inference: {len(ogg_files)}"
)

if not ogg_files:
    print(
        f"WARNING: no required .ogg under {TRAIN_SOUNDSCAPES_DIR}. "
        "Download/train_soundscapes data is required to run inference."
    )

results_rows: list[dict] = []

for mi, mp in enumerate(eval_paths):
    label = mp.parent.name
    print(f"[{mi + 1}/{len(eval_paths)}] Loading {label} …")
    model = tf.keras.models.load_model(mp)

    pred_df = soundscapes_inference_df(
        model,
        ogg_files,
        species_cols,
        progress_every=10,
    )

    pred_renamed = pred_df.rename(columns={c: f"{c}_pred" for c in species_cols})
    merged = y_true_df.reset_index().merge(pred_renamed, on="row_id", how="inner")
    if merged.empty:
        print("  ERROR: no overlapping row_id — check filenames vs labels CSV.")
        continue

    pred_cols = [f"{c}_pred" for c in species_cols]
    yt = merged[species_cols].to_numpy(dtype=np.float32)
    yp = merged[pred_cols].to_numpy(dtype=np.float32)

    pos_counts = yt.sum(axis=0)
    n_species_pos = int((pos_counts > 0).sum())

    auc_ge1 = macro_auc_filtered(yt, yp, min_pos=1)
    auc_ge3 = macro_auc_filtered(yt, yp, min_pos=3)
    t5 = top5_hit_rate(yt, yp)
    map_macro = macro_map(yt, yp)

    results_rows.append(
        {
            "submission_label": label,
            "model_path": str(mp),
            "macro_auc_ge1_pos": auc_ge1,
            "macro_auc_ge3_pos": auc_ge3,
            "top5_hit_rate": t5,
            "macro_map": map_macro,
            "n_windows_eval": len(merged),
            "n_species_with_positive": n_species_pos,
            "actual_lb_score": np.nan,
        }
    )
    print(
        f"  macro_AUC (≥1 pos): {auc_ge1:.4f} | macro_AUC (≥3 pos): {auc_ge3:.4f} | "
        f"top5: {t5:.4f} | mAP: {map_macro:.4f} | windows: {len(merged)}"
    )

results_df = pd.DataFrame(results_rows)
results_df

[1/1] Loading sub_5_29.04_v1.4 …


/Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  … 10/10658 soundscapes
  … 20/10658 soundscapes
  … 30/10658 soundscapes
  … 40/10658 soundscapes
  … 50/10658 soundscapes
  … 60/10658 soundscapes
  … 70/10658 soundscapes
  … 80/10658 soundscapes
  … 90/10658 soundscapes
  … 100/10658 soundscapes
  … 110/10658 soundscapes
  … 120/10658 soundscapes
  … 130/10658 soundscapes
  … 140/10658 soundscapes
  … 150/10658 soundscapes
  … 160/10658 soundscapes
  … 170/10658 soundscapes
  … 180/10658 soundscapes
  … 190/10658 soundscapes
  … 200/10658 soundscapes
  … 210/10658 soundscapes
  … 220/10658 soundscapes
  … 230/10658 soundscapes
  … 240/10658 soundscapes
  … 250/10658 soundscapes
  … 260/10658 soundscapes
  … 270/10658 soundscapes
  … 280/10658 soundscapes
  … 290/10658 soundscapes
  … 300/10658 soundscapes
  … 310/10658 soundscapes
  … 320/10658 soundscapes
  … 330/10658 soundscapes
  … 340/10658 soundscapes
  … 350/10658 soundscapes
  … 360/10658 soundscapes
  … 370/10658 soundscapes
  … 380/10658 soundscapes
  … 390/10658 soundsc

,submission_label,model_path,macro_auc_ge1_pos,macro_auc_ge3_pos,top5_hit_rate,macro_map,n_windows_eval,n_species_with_positive,actual_lb_score
0,sub_5_29.04_v1.4,/Users/enricozafiris/Desktop/Catolica/T4 Advan...,0.688521,0.690177,0.322057,0.066339,739,75,NaN


## Results table

Fill **`actual_lb_score`** manually with your recorded leaderboard macro-AUC (or related official score) for each submission row, then run the comparison cell below.

You can edit **`results_df`** in-place or reload from CSV after annotating.

In [ ]:
# Example manual LB annotation (replace with your values):
# lb_map = {
#     "sub_9_30.04_v2.0_pretrained": 0.72,
#     "sub_8_30.04_v1.7_sec_label_weighted_0_5": 0.68,
# }
# results_df["actual_lb_score"] = results_df["submission_label"].map(lb_map)

results_df

## Comparison — Spearman correlation vs leaderboard

After **`actual_lb_score`** is populated, this cell computes rank correlation between **`macro_auc_ge1_pos`** (and **`macro_auc_ge3_pos`**) and LB scores, and plots **local soundscape AUC** vs **LB** with the **y = x** reference line.

Adjust **`METRIC_COL`** if your leaderboard metric aligns better with the **`≥3`** positive subset.

In [ ]:
METRIC_COL = "macro_auc_ge1_pos"

cmp = results_df.dropna(subset=["actual_lb_score"]).copy()
if len(cmp) < 2:
    print("Need at least 2 rows with non-null actual_lb_score for correlation.")
else:
    rho, pval = spearmanr(cmp[METRIC_COL], cmp["actual_lb_score"])
    rho3, _ = spearmanr(cmp["macro_auc_ge3_pos"], cmp["actual_lb_score"])
    print(f"Spearman ρ ({METRIC_COL} vs LB): {rho:.3f} (p≈{pval:.3g})")
    print(f"Spearman ρ (macro_auc_ge3_pos vs LB): {rho3:.3f}")

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(cmp[METRIC_COL], cmp["actual_lb_score"], s=48)
    for _, r in cmp.iterrows():
        ax.annotate(
            r["submission_label"],
            (r[METRIC_COL], r["actual_lb_score"]),
            fontsize=7,
            alpha=0.85,
        )
    lo = float(min(cmp[METRIC_COL].min(), cmp["actual_lb_score"].min()))
    hi = float(max(cmp[METRIC_COL].max(), cmp["actual_lb_score"].max()))
    pad = (hi - lo) * 0.05 + 1e-6
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], "k--", lw=1, label="y = x")
    ax.set_xlabel(f"Local soundscape {METRIC_COL}")
    ax.set_ylabel("Actual leaderboard score")
    ax.legend()
    ax.set_aspect("equal", adjustable="box")
    plt.tight_layout()
    plt.show()

## How to interpret

- **Strong evidence for soundscape validation:** Spearman **ρ > ~0.7** between **`macro_auc_ge1_pos`** (or **`macro_auc_ge3_pos`**) and **`actual_lb_score`**, *and* points lie near the **y = x** diagonal (typical gap **≤ ~0.03** if scales match). Then local scoring on **`train_soundscapes/`** is likely a trustworthy proxy for leaderboard ranking—reasonable to refactor training so validation tracks this distribution.
- **Weak or unstable correlation:** LB vs local disagreement suggests remaining gaps—e.g. **domain shift** still present between labeled train soundscapes and test, **preprocessing mismatch** vs the notebook used on Kaggle, **different checkpoint** than uploaded **`model.keras`**, or **metric definition** differences vs official LB aggregation.
- **Absolute scale:** LB AUC and train-soundscape AUC need not match numerically if the hidden test set differs in difficulty; **rank correlation** plus qualitative diagonal closeness is usually the decisive sanity check.

Use this notebook’s outcome only as **supporting evidence**: correlation across a handful of submissions is noisy; add more archived checkpoints when possible.